In [46]:
import pandas as pd
import numpy as np

np.random.seed(42)

In [47]:
# -- TABLE 1: Defining the Asset Metadata (Table 1: dim_aircraft)

aircraft_list = [f'N{i:03d}AL' for i in range(101, 111)]
dim_aircraft = pd.DataFrame({
    'tail_number': aircraft_list,
    'model': 'Alice_V1',
    'battery_capacity_kwh': 850, 
    'max_payload_lbs': 2500,
    'firmware_version': np.random.choice(['v2.1', 'v2.2'], 10)
})

dim_aircraft.to_csv('aviation_data/dim_aircraft.csv', index=False)
dim_aircraft.head()

,tail_number,model,battery_capacity_kwh,max_payload_lbs,firmware_version
0,N101AL,Alice_V1,850,2500,v2.1
1,N102AL,Alice_V1,850,2500,v2.2
2,N103AL,Alice_V1,850,2500,v2.1
3,N104AL,Alice_V1,850,2500,v2.1
4,N105AL,Alice_V1,850,2500,v2.1


In [48]:
print(dim_aircraft.shape)

(10, 5)


In [49]:
# --- TABLE 2: Flight_Summary (Mission Metadata) 

num_flights = 100 

# 2. Mission Data
flight_summary = pd.DataFrame({
    'flight_id': [f'FL-{i:05d}' for i in range(1, num_flights + 1)],
    
    # Randomly assign an aircraft from our fleet (Link to Table 1)
    'tail_number': np.random.choice(dim_aircraft['tail_number'], num_flights),
    
    # Commercial Load: Cargo weight between 500 lbs and the 2500 lbs limit
    'payload_lbs': np.random.uniform(500, 2500, num_flights),
    
    # Environmental factors: Temperature affects battery efficiency significantly
    'ambient_temp_c': np.random.uniform(-5, 45, num_flights),
    
    # Strategic Revenue data for ROI modeling
    'ticket_revenue': np.random.uniform(3000, 7000, num_flights)
})


flight_summary.to_csv('aviation_data/flight_summary.csv', index=False)
flight_summary.head()

,flight_id,tail_number,payload_lbs,ambient_temp_c,ticket_revenue
0,FL-00001,N103AL,1959.212357,35.861110,4862.392073
1,FL-00002,N107AL,1775.114943,22.760041,5170.578539
2,FL-00003,N108AL,2274.425485,21.482529,4146.165009
3,FL-00004,N105AL,1444.429850,7.092615,5363.333042
4,FL-00005,N104AL,739.188492,-0.344862,3122.001000


In [50]:
print(flight_summary.shape)

(100, 5)


In [51]:
# --- Table 3: Fact_Telemetry_HighFreq (w/ Official Spec Alignment)

telemetry_list = [] 
total_duration = 3000 
#50 minutes to reflect 250nm range potential

# 1. Each flight is an independent, isolate each flight_id to apply its specific payload (weight) and ambient_temp (temperature)

for index, flight in flight_summary.iterrows():
    f_id = flight['flight_id']
    payload = flight['payload_lbs']
    temp = flight['ambient_temp_c']
    
    payload_factor = (payload / 2500) * 0.0006
    temp_factor = abs(temp - 25) * 0.00003
    current_soc = 100.0
    
    # Use conditional logic (If/Else) that changes the aircraft's state based on the current time (by second)
    for second in range(total_duration):
        
        if second < 450:
            phase, altitude, phase_load = 'CLIMB', second * 33.3, 0.025
        elif second < 2500:
            phase, altitude, phase_load = 'CRUISE', 15000, 0.012
        else:
            phase, altitude, phase_load = 'LANDING', 15000 - ((second - 2500) * 30), 0.008

        drain = (phase_load + payload_factor + temp_factor) / 10
        current_soc -= drain
        
        # append "for second"
        telemetry_list.append({
            'flight_id': f_id,
            'timestamp_sec': second,
            'altitude_ft': max(0, altitude),
            'battery_soc_pct': round(current_soc, 2),
            'phase': phase
        })

fact_telemetry_high_freq = pd.DataFrame(telemetry_list)
fact_telemetry_high_freq.to_csv('aviation_data/fact_telemetry_high_freq.csv', index=False)
fact_telemetry_high_freq.sample(10)

,flight_id,timestamp_sec,altitude_ft,battery_soc_pct,phase
145873,FL-00049,1873,15000.0,96.95,CRUISE
70412,FL-00024,1412,15000.0,97.60,CRUISE
67649,FL-00023,1649,15000.0,97.25,CRUISE
19487,FL-00007,1487,15000.0,97.49,CRUISE
242373,FL-00081,2373,15000.0,96.47,CRUISE
117127,FL-00040,127,4229.1,99.67,CLIMB
236749,FL-00079,2749,7530.0,96.06,LANDING
282958,FL-00095,958,15000.0,98.20,CRUISE
281524,FL-00094,2524,14280.0,96.27,LANDING
116093,FL-00039,2093,15000.0,96.75,CRUISE


In [52]:
{len(fact_telemetry_high_freq)}

{300000}